In [ ]:
!pip install -q sentence-transformers qdrant-client python-dotenv pypdf pyyaml

import os
import shutil
import sys

SRC = "/kaggle/input/datasets/nourhene65/lingua-ar"
DST = "/kaggle/working/Lingua-rag"
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
os.chdir(DST)
sys.path.insert(0, DST)
print(os.listdir("."))
print(os.listdir("ar/src"))
print(os.path.exists("ar/data_derived/pairs_ft.jsonl"))

In [ ]:
import torch

print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
from ar.src import finetune as ft
from ar.src.index import load_chunks, load_config

config = load_config()
config["qdrant_url"] = ":memory:"  # evaluer_ft gere ce mode

chunks = load_chunks()
valides = {ft._cle(c["texte"].strip()) for c in chunks}
paires = [p for p in ft._lire_paires().values() if p["cle"] in valides]
print("paires utilisables :", len(paires))

ft.entrainer(config, paires, 42)
apres, echecs = ft.evaluer_ft(config, chunks, ft._load_questions(ft.QUESTIONS))
for groupe, m in apres.items():
    print(groupe, m)
print("echecs :", echecs)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

from ar.src.ingest import normalize

questions = ft._load_questions(ft.QUESTIONS)


def rangs(chemin, normaliser):
    m = SentenceTransformer(str(chemin))
    P = m.encode(
        ["passage: " + c["texte"] for c in chunks],
        normalize_embeddings=True,
        batch_size=16,
        show_progress_bar=False,
    )
    out = []
    for q in questions:
        t = normalize(q["question"], "ar") if normaliser else q["question"]
        qv = m.encode(["query: " + t], normalize_embeddings=True)[0]
        top = np.argsort(-(P @ qv))[:10]
        pas = [
            {
                "doc_id": chunks[i]["doc_id"],
                "page_ou_section": chunks[i]["page_ou_section"],
            }
            for i in top
        ]
        out.append(ft._rank(pas, q))
    return out


res = {}
for nom, chemin in [
    ("base", config_base := ft.model_name(config)),
    ("ft", ft.MODEL_DIR),
]:
    for norm in (False, True):
        r = rangs(chemin, norm)
        res[(nom, norm)] = r
        for groupe in ("meme_langue", "translingue"):
            sel = [x for x, q in zip(r, questions) if q["type"] == groupe]
            print(nom, "normalise" if norm else "brut", groupe, ft._metrics(sel))

print("\nq   type          base  ft")
for i, q in enumerate(questions, 1):
    print(
        f"q{i:02d} {q['type']:12s} {res[('base', True)][i - 1]}  {res[('ft', True)][i - 1]}"
    )

In [ ]:
def charger(chemin):
    m = SentenceTransformer(str(chemin))
    P = m.encode(
        ["passage: " + c["texte"] for c in chunks],
        normalize_embeddings=True,
        batch_size=16,
        show_progress_bar=False,
    )
    return m, P


def classer(m, P, q, ar_seulement=False):
    qv = m.encode(
        ["query: " + normalize(q["question"], "ar")], normalize_embeddings=True
    )[0]
    s = P @ qv
    idx = list(np.argsort(-s))
    if ar_seulement:
        idx = [i for i in idx if chunks[i]["langue"] == "ar"]
    return idx[:10], s


modeles = {"base": charger(ft.model_name(config)), "ft": charger(ft.MODEL_DIR)}

# 1) Qui passe devant le bon passage ?
m, P = modeles["ft"]
for n in (5, 6, 8, 12, 15):
    q = questions[n - 1]
    idx, s = classer(m, P, q)
    print(
        f"\nq{n:02d} attendu : {q['doc_id_attendu']} p.{q['page_ou_section_attendue']}"
    )
    for r, i in enumerate(idx[:5], 1):
        c = chunks[i]
        print(f"  {r}. {c['doc_id']} p.{c['page_ou_section']}  {s[i]:.3f}")

# 2) Même langue, en cherchant seulement dans les documents arabes
for nom, (m, P) in modeles.items():
    r = []
    for q in questions:
        if q["type"] == "meme_langue":
            idx, _ = classer(m, P, q, ar_seulement=True)
            pas = [
                {
                    "doc_id": chunks[i]["doc_id"],
                    "page_ou_section": chunks[i]["page_ou_section"],
                }
                for i in idx
            ]
            r.append(ft._rank(pas, q))
    print(nom, "meme_langue, docs arabes seulement :", ft._metrics(r))

In [ ]:
q = questions[14]
print(q["question"], "| attendu :", q["doc_id_attendu"], q["page_ou_section_attendue"])
for c in chunks:
    if c["doc_id"] == "physical-activity_en" and str(c["page_ou_section"]) == "1":
        print("\n--- p.1 ---\n", c["texte"][:400])
idx, s = classer(*modeles["ft"], q)
print("\n--- rang 1 ---\n", chunks[idx[0]]["texte"][:400])

In [ ]:
!cd ar/models && zip -qr /kaggle/working/e5_ft.zip e5_ft
!ls -lh /kaggle/working

In [ ]:
!ls -lh /kaggle/working